<a href="https://colab.research.google.com/github/id25s017-blip/AE5510/blob/main/AE5510_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/AE5510"   # EDIT THIS if you want a different folder
os.makedirs(DRIVE_DIR, exist_ok=True)
print("DRIVE_DIR ready:", DRIVE_DIR)

Mounted at /content/drive
DRIVE_DIR ready: /content/drive/MyDrive/AE5510


In [ ]:
import pandas as pd

# EDIT THIS to your actual CSV path
CSV_PATH = "/content/drive/MyDrive/swarm_criticality_dataset_100_scenarios.csv"

df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df.columns.tolist())

(1639166, 37)
['case_id', 'num_agents', 'n_obstacles', 'obstacle_density', 'mean_obstacle_size', 'clutter_index', 'obstacle_clustering', 'attrition_rate', 'step', 'agent_id', 'rel_x', 'rel_y', 'rel_z', 'dist_to_centroid', 'rank_dist_to_centroid', 'dist_to_target', 'rank_dist_to_target', 'vx', 'vy', 'vz', 'speed', 'heading', 'flight_path_angle', 'accel', 'heading_rate', 'num_neighbors', 'mean_neighbor_dist', 'min_neighbor_dist', 'weighted_density', 'angular_coverage', 'criticality_score', 'degree_centrality', 'closeness_centrality', 'betweenness_centrality', 'eigenvector_centrality', 'is_critical', 'was_promoted']


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def parse_case_id(case_id):
    # "c001_n60_none" -> (60, "none")
    parts = case_id.split("_")
    n_agents = int(parts[1][1:])       # strip leading "n"
    attrition = parts[2]
    return n_agents, attrition

all_case_ids = sorted(df["case_id"].unique())          # should be 100
case_info = pd.DataFrame({"case_id": all_case_ids})
case_info[["n_agents", "attrition"]] = case_info["case_id"].apply(
    lambda c: pd.Series(parse_case_id(c))
)

# Don't stratify on the exact n_agents value -- swarm sizes vary enough that most
# n_agents+attrition combinations are unique to a single scenario. Bucket n_agents into
# a handful of ranges first, then stratify on bucket+attrition instead.
case_info["size_bucket"] = pd.qcut(case_info["n_agents"], q=4, duplicates="drop").astype(str)
case_info["strata"] = case_info["size_bucket"] + "_" + case_info["attrition"]
print(case_info["strata"].value_counts())   # sanity check: every group should have >= 2 scenarios

def safe_split(ids, strata, test_size, random_state):
    """Stratify if every group has >=2 members; otherwise fall back to a plain shuffle."""
    if strata.value_counts().min() >= 2:
        return train_test_split(ids, test_size=test_size, random_state=random_state, stratify=strata)
    print("[split] some strata still have <2 members -- falling back to an unstratified shuffle")
    return train_test_split(ids, test_size=test_size, random_state=random_state)

train_ids, holdout_ids = safe_split(
    case_info["case_id"], case_info["strata"], test_size=0.30, random_state=42
)
val_ids, test_ids = safe_split(
    holdout_ids, case_info.set_index("case_id").loc[holdout_ids, "strata"],
    test_size=0.50, random_state=42,
)

print(f"train scenarios: {len(train_ids)} | val scenarios: {len(val_ids)} | test scenarios: {len(test_ids)}")

strata
(28.5, 45.0]_medium      13
(17.999, 28.5]_heavy      9
(60.0, 75.0]_light        7
(45.0, 60.0]_none         7
(60.0, 75.0]_heavy        7
(28.5, 45.0]_none         7
(28.5, 45.0]_light        6
(28.5, 45.0]_heavy        6
(17.999, 28.5]_light      6
(45.0, 60.0]_light        5
(60.0, 75.0]_medium       5
(17.999, 28.5]_none       5
(60.0, 75.0]_none         5
(17.999, 28.5]_medium     5
(45.0, 60.0]_medium       4
(45.0, 60.0]_heavy        3
Name: count, dtype: int64
[split] some strata still have <2 members -- falling back to an unstratified shuffle
train scenarios: 70 | val scenarios: 15 | test scenarios: 15


In [ ]:
import torch
from torch_geometric.data import Data
from scipy.spatial import cKDTree

SENSING_RANGE = 55.0

# Position relative to the swarm centroid, 3D -- confirmed from your real schema.
POS_COLS = ["rel_x", "rel_y", "rel_z"]

# Tier 1's 14 features -- confirm against your actual LightGBM feature list if you have it;
# this is the best-guess default (3 connectivity + 4 centrality + 1 attrition + 6 kinematics).
FEATURE_COLUMNS = [
    "min_neighbor_dist", "weighted_density", "angular_coverage",            # connectivity (3)
    "degree_centrality", "closeness_centrality",
    "betweenness_centrality", "eigenvector_centrality",                      # centrality (4)
    "attrition_rate",                                                       # ordinal (1)
    "vx", "vy", "vz", "speed", "heading", "accel",                         # kinematics (6)
]
assert len(FEATURE_COLUMNS) == 14, "should match the 14 features from the Second Review deck"

# Leakage guard: is_critical is the prediction target, never a model input; criticality_score
# is only used post-hoc (Section 8's top-k ranking check), never a model input either. If either
# of these ever ends up in FEATURE_COLUMNS, that's a bug -- fail loudly instead of training on it.
assert "is_critical" not in FEATURE_COLUMNS, \
    "is_critical must never be a model input -- it is the prediction target"
assert "criticality_score" not in FEATURE_COLUMNS, \
    "criticality_score must never be a model input -- it's reported post-hoc only, this would be leakage"

ATTRITION_ORDER = {"none": 0, "light": 1, "medium": 2, "heavy": 3}

def build_graphs(df, case_ids, feature_columns=FEATURE_COLUMNS, pos_cols=POS_COLS,
                  sensing_range=SENSING_RANGE):
    # Same leakage guard, re-checked here so it still catches a mistake even if someone edits
    # feature_columns right before calling this function, bypassing the module-level checks above.
    assert "is_critical" not in feature_columns and "criticality_score" not in feature_columns, \
        "is_critical/criticality_score must never appear in the model's input features"

    # Self-contained: encodes attrition_rate if it isn't already, and swaps it into the
    # feature list if it's still the raw name -- works regardless of cell execution order.
    feature_columns = list(feature_columns)  # local copy -- never mutate the caller's list
    if "attrition_rate_ord" not in df.columns:
        df["attrition_rate_ord"] = df["attrition_rate"].map(ATTRITION_ORDER)
        assert df["attrition_rate_ord"].isna().sum() == 0, \
            "found an attrition_rate value outside none/light/medium/heavy"
    if "attrition_rate" in feature_columns:
        feature_columns[feature_columns.index("attrition_rate")] = "attrition_rate_ord"

    graphs = []
    subset = df[df["case_id"].isin(case_ids)]
    for (case_id, step), g in subset.groupby(["case_id", "step"], sort=False):
        g = g.reset_index(drop=True)
        if len(g) < 2:
            continue  # need at least 2 agents for an edge to exist

        pos = g[pos_cols].to_numpy(dtype=np.float32)
        pairs = cKDTree(pos).query_pairs(r=sensing_range, output_type="ndarray")
        if len(pairs) == 0:
            continue  # fully disconnected snapshot -- skip

        edges = np.concatenate([pairs, pairs[:, ::-1]], axis=0)       # undirected: both directions
        edge_index = torch.tensor(edges.T, dtype=torch.long)          # shape [2, num_edges]

        x = torch.tensor(g[feature_columns].to_numpy(dtype=np.float32))
        y = torch.tensor(g["is_critical"].to_numpy(dtype=np.float32))

        data = Data(x=x, edge_index=edge_index, y=y)
        data.case_id = case_id
        data.step = int(step)
        data.criticality_score = torch.tensor(g["criticality_score"].to_numpy(dtype=np.float32))
        data.clutter_index = float(g["clutter_index"].iloc[0])
        graphs.append(data)
    return graphs

train_graphs = build_graphs(df, train_ids)
val_graphs   = build_graphs(df, val_ids)
test_graphs  = build_graphs(df, test_ids)
print(f"{len(train_graphs)} train graphs | {len(val_graphs)} val graphs | {len(test_graphs)} test graphs")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


27152 train graphs | 5228 val graphs | 5351 test graphs


In [ ]:
sample_case, sample_step = train_ids.iloc[0], 0
g = df[(df["case_id"] == sample_case) & (df["step"] == sample_step)].reset_index(drop=True)
pos = g[POS_COLS].to_numpy(dtype=np.float32)
dists, _ = cKDTree(pos).query(pos, k=2)          # k=2: nearest neighbor excluding self
computed_min_dist = dists[:, 1]

check = pd.DataFrame({
    "dataset_min_neighbor_dist": g["min_neighbor_dist"],
    "computed_min_dist": computed_min_dist,
})
check["abs_diff"] = (check["dataset_min_neighbor_dist"] - check["computed_min_dist"]).abs()
print(check["abs_diff"].describe())   # should be ~0 if POS_COLS and units are right

count    2.400000e+01
mean     4.690028e-07
std      3.233910e-07
min      4.327031e-08
25%      1.779897e-07
50%      5.049379e-07
75%      7.197805e-07
max      1.040429e-06
Name: abs_diff, dtype: float64


In [ ]:
torch.save(train_graphs, f"{DRIVE_DIR}/gnn_train_graphs.pt")
torch.save(val_graphs,   f"{DRIVE_DIR}/gnn_val_graphs.pt")
torch.save(test_graphs,  f"{DRIVE_DIR}/gnn_test_graphs.pt")
print("cached to", DRIVE_DIR)

# To reload in a fresh runtime instead of rebuilding (after re-running the Setup cells above):
# train_graphs = torch.load(f"{DRIVE_DIR}/gnn_train_graphs.pt", weights_only=False)
# val_graphs   = torch.load(f"{DRIVE_DIR}/gnn_val_graphs.pt",   weights_only=False)
# test_graphs  = torch.load(f"{DRIVE_DIR}/gnn_test_graphs.pt",  weights_only=False)

cached to /content/drive/MyDrive/AE5510


In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_graphs,  batch_size=64, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class InductiveGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.lin = nn.Linear(hidden_channels, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x).squeeze(-1)   # one raw logit per node

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

n_pos = sum(int(g.y.sum().item()) for g in train_graphs)
n_total = sum(g.num_nodes for g in train_graphs)
pos_weight = torch.tensor([(n_total - n_pos) / max(n_pos, 1)], device=device)
print(f"train set: {n_total} nodes, {n_pos} critical ({100*n_pos/n_total:.1f}%), pos_weight={pos_weight.item():.2f}")

model = InductiveGNN(in_channels=len(FEATURE_COLUMNS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

from sklearn.metrics import average_precision_score

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index)
            loss = criterion(logits, batch.y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_auc_pr(loader):
    model.eval()
    all_logits, all_y = [], []
    for batch in loader:
        batch = batch.to(device)
        all_logits.append(model(batch.x, batch.edge_index).cpu())
        all_y.append(batch.y.cpu())
    probs = torch.sigmoid(torch.cat(all_logits)).numpy()
    y = torch.cat(all_y).numpy()
    return average_precision_score(y, probs)

best_val_auc_pr, patience, bad_epochs = -1.0, 10, 0
for epoch in range(1, 101):
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader, train=False)
    val_auc_pr = eval_auc_pr(val_loader)
    print(f"epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val AUC-PR {val_auc_pr:.4f}")

    if val_auc_pr > best_val_auc_pr:
        best_val_auc_pr, bad_epochs = val_auc_pr, 0
        torch.save(model.state_dict(), f"{DRIVE_DIR}/gnn_best_checkpoint.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("early stopping")
            break

train set: 1190385 nodes, 465412 critical (39.1%), pos_weight=1.56
epoch 001 | train_loss 0.3933 | val_loss 0.2871 | val AUC-PR 0.9661
epoch 002 | train_loss 0.2868 | val_loss 0.2545 | val AUC-PR 0.9744
epoch 003 | train_loss 0.2605 | val_loss 0.2229 | val AUC-PR 0.9791
epoch 004 | train_loss 0.2452 | val_loss 0.2155 | val AUC-PR 0.9809
epoch 005 | train_loss 0.2345 | val_loss 0.2023 | val AUC-PR 0.9821
epoch 006 | train_loss 0.2268 | val_loss 0.1991 | val AUC-PR 0.9834
epoch 007 | train_loss 0.2201 | val_loss 0.1885 | val AUC-PR 0.9846
epoch 008 | train_loss 0.2150 | val_loss 0.1817 | val AUC-PR 0.9858
epoch 009 | train_loss 0.2118 | val_loss 0.1828 | val AUC-PR 0.9856
epoch 010 | train_loss 0.2074 | val_loss 0.1807 | val AUC-PR 0.9858
epoch 011 | train_loss 0.2050 | val_loss 0.1743 | val AUC-PR 0.9868
epoch 012 | train_loss 0.2026 | val_loss 0.1801 | val AUC-PR 0.9865
epoch 013 | train_loss 0.2017 | val_loss 0.1730 | val AUC-PR 0.9866
epoch 014 | train_loss 0.1996 | val_loss 0.1731 |

In [ ]:
import itertools, random

param_grid = {
    "hidden_channels": [32, 64, 128],
    "num_layers": [2, 3],
    "lr": [1e-3, 5e-4],
    "dropout": [0.2, 0.3, 0.5],
}
all_combos = list(itertools.product(*param_grid.values()))
random.seed(42)
sampled_combos = random.sample(all_combos, k=10)

def train_one_config(hidden_channels, num_layers, lr, dropout, max_epochs=40, patience=6):
    cfg_model = InductiveGNN(len(FEATURE_COLUMNS), hidden_channels, num_layers, dropout).to(device)
    cfg_optimizer = torch.optim.Adam(cfg_model.parameters(), lr=lr, weight_decay=1e-5)
    cfg_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best, bad = -1.0, 0
    for epoch in range(max_epochs):
        cfg_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            cfg_optimizer.zero_grad()
            loss = cfg_criterion(cfg_model(batch.x, batch.edge_index), batch.y)
            loss.backward()
            cfg_optimizer.step()
        cfg_model.eval()
        with torch.no_grad():
            logits, ys = [], []
            for batch in val_loader:
                batch = batch.to(device)
                logits.append(cfg_model(batch.x, batch.edge_index).cpu())
                ys.append(batch.y.cpu())
            val_score = average_precision_score(torch.cat(ys), torch.sigmoid(torch.cat(logits)))
        if val_score > best:
            best, bad = val_score, 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best

results = []
for hc, nl, lr, dp in sampled_combos:
    score = train_one_config(hc, nl, lr, dp)
    results.append({"hidden_channels": hc, "num_layers": nl, "lr": lr, "dropout": dp, "val_auc_pr": score})
    print(results[-1])

results_df = pd.DataFrame(results).sort_values("val_auc_pr", ascending=False)
print(results_df)

{'hidden_channels': 32, 'num_layers': 3, 'lr': 0.001, 'dropout': 0.3, 'val_auc_pr': np.float64(0.9856507267335719)}
{'hidden_channels': 32, 'num_layers': 2, 'lr': 0.001, 'dropout': 0.3, 'val_auc_pr': np.float64(0.985459316321704)}
{'hidden_channels': 64, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.5, 'val_auc_pr': np.float64(0.983604038539818)}
{'hidden_channels': 64, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.2, 'val_auc_pr': np.float64(0.9888674854063305)}
{'hidden_channels': 64, 'num_layers': 2, 'lr': 0.001, 'dropout': 0.5, 'val_auc_pr': np.float64(0.9864481608177135)}
{'hidden_channels': 32, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.3, 'val_auc_pr': np.float64(0.9836896480615702)}
{'hidden_channels': 64, 'num_layers': 3, 'lr': 0.0005, 'dropout': 0.5, 'val_auc_pr': np.float64(0.9873325360490882)}
{'hidden_channels': 32, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.2, 'val_auc_pr': np.float64(0.984298912709033)}
{'hidden_channels': 64, 'num_layers': 3, 'lr': 0.0005, 'dropout': 0.2,

In [ ]:
best_cfg = results_df.iloc[0]
hidden_channels = int(best_cfg.hidden_channels)
num_layers = int(best_cfg.num_layers)
lr = float(best_cfg.lr)
dropout = float(best_cfg.dropout)
print(f"retraining with: hidden_channels={hidden_channels}, num_layers={num_layers}, lr={lr}, dropout={dropout}")

model = InductiveGNN(len(FEATURE_COLUMNS), hidden_channels, num_layers, dropout).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_auc_pr, patience, bad_epochs = -1.0, 10, 0
for epoch in range(1, 101):
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader, train=False)
    val_auc_pr = eval_auc_pr(val_loader)
    print(f"epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val AUC-PR {val_auc_pr:.4f}")
    if val_auc_pr > best_val_auc_pr:
        best_val_auc_pr, bad_epochs = val_auc_pr, 0
        torch.save(model.state_dict(), f"{DRIVE_DIR}/gnn_best_checkpoint.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("early stopping")
            break

retraining with: hidden_channels=64, num_layers=2, lr=0.0005, dropout=0.2
epoch 001 | train_loss 0.4197 | val_loss 0.3131 | val AUC-PR 0.9590
epoch 002 | train_loss 0.3020 | val_loss 0.2678 | val AUC-PR 0.9704
epoch 003 | train_loss 0.2693 | val_loss 0.2410 | val AUC-PR 0.9753
epoch 004 | train_loss 0.2491 | val_loss 0.2269 | val AUC-PR 0.9783
epoch 005 | train_loss 0.2354 | val_loss 0.2109 | val AUC-PR 0.9809
epoch 006 | train_loss 0.2258 | val_loss 0.2061 | val AUC-PR 0.9823
epoch 007 | train_loss 0.2184 | val_loss 0.1958 | val AUC-PR 0.9833
epoch 008 | train_loss 0.2121 | val_loss 0.1929 | val AUC-PR 0.9843
epoch 009 | train_loss 0.2077 | val_loss 0.1844 | val AUC-PR 0.9849
epoch 010 | train_loss 0.2030 | val_loss 0.1782 | val AUC-PR 0.9857
epoch 011 | train_loss 0.2000 | val_loss 0.1801 | val AUC-PR 0.9857
epoch 012 | train_loss 0.1967 | val_loss 0.1772 | val AUC-PR 0.9861
epoch 013 | train_loss 0.1939 | val_loss 0.1766 | val AUC-PR 0.9868
epoch 014 | train_loss 0.1910 | val_loss 0

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

@torch.no_grad()
def detailed_eval(model, graphs):
    model.eval()
    rows = []
    for g in graphs:
        data = g.to(device)
        probs = torch.sigmoid(model(data.x, data.edge_index)).cpu().numpy()
        y = data.y.cpu().numpy()
        n_agents, attrition = parse_case_id(g.case_id)
        rows.append(pd.DataFrame({
            "case_id": g.case_id, "step": g.step,
            "n_agents": n_agents, "attrition": attrition,
            "clutter_index": g.clutter_index,
            "y_true": y, "y_prob": probs,
            "criticality_score": g.criticality_score.numpy(),
        }))
    return pd.concat(rows, ignore_index=True)

model.load_state_dict(torch.load(f"{DRIVE_DIR}/gnn_best_checkpoint.pt"))
detail_df = detailed_eval(model, test_graphs)

In [ ]:
from sklearn.metrics import precision_recall_curve

@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    probs, ys = [], []
    for batch in loader:
        batch = batch.to(device)
        probs.append(torch.sigmoid(model(batch.x, batch.edge_index)).cpu())
        ys.append(batch.y.cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

val_probs, val_y = collect_probs(model, val_loader)
precisions, recalls, thresholds = precision_recall_curve(val_y, val_probs)

# lowest threshold that still keeps precision >= 0.80 -- maximizes recall subject to that floor.
candidates = [(t, p) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds) if p >= 0.80]
threshold = min(candidates, key=lambda x: x[0])[0] if candidates else 0.5
print(f"chosen threshold: {threshold:.3f}")

detail_df["y_pred"] = (detail_df["y_prob"] >= threshold).astype(int)

chosen threshold: 0.116


**Vertical 1 — overall held-out performance:**

In [ ]:
overall = {
    "accuracy":  accuracy_score(detail_df.y_true, detail_df.y_pred),
    "precision": precision_score(detail_df.y_true, detail_df.y_pred),
    "recall":    recall_score(detail_df.y_true, detail_df.y_pred),
    "f1":        f1_score(detail_df.y_true, detail_df.y_pred),
    "auc_pr":    average_precision_score(detail_df.y_true, detail_df.y_prob),
}
print(overall)

{'accuracy': 0.8681729094593376, 'precision': 0.746981434615459, 'recall': 0.9799156075370326, 'f1': 0.8477387458000581, 'auc_pr': np.float64(0.9766799582342885)}


**Vertical 2 — swarm size:**

In [ ]:
by_swarm_size = detail_df.groupby(pd.cut(detail_df.n_agents, bins=[0, 40, 70, 100, 200])).apply(
    lambda d: f1_score(d.y_true, d.y_pred)
).rename("f1")
print(by_swarm_size)

n_agents
(0, 40]       0.817252
(40, 70]      0.846786
(70, 100]     0.954907
(100, 200]    0.000000
Name: f1, dtype: float64


/tmp/ipykernel_485/301546771.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_swarm_size = detail_df.groupby(pd.cut(detail_df.n_agents, bins=[0, 40, 70, 100, 200])).apply(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Vertical 3 — obstacle / clutter layout:**

In [ ]:
by_clutter = detail_df.groupby(pd.qcut(detail_df.clutter_index, q=4, duplicates="drop")).apply(
    lambda d: f1_score(d.y_true, d.y_pred)
).rename("f1")
print(by_clutter)

clutter_index
(-0.001, 0.306]    0.847260
(0.306, 0.511]     0.870389
(0.511, 0.8]       0.812053
(0.8, 1.0]         0.940055
Name: f1, dtype: float64


/tmp/ipykernel_485/3499009700.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_clutter = detail_df.groupby(pd.qcut(detail_df.clutter_index, q=4, duplicates="drop")).apply(


**Vertical 4 — attrition level:**

In [ ]:
by_attrition = detail_df.groupby("attrition").apply(
    lambda d: f1_score(d.y_true, d.y_pred)
).rename("f1")
print(by_attrition)

attrition
heavy     0.770688
light     0.835792
medium    0.898009
none      0.903194
Name: f1, dtype: float64


/tmp/ipykernel_485/2192852958.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_attrition = detail_df.groupby("attrition").apply(


**Vertical 5 — ranking quality (top-k), matching Second Review slide 7:**

In [ ]:
top20 = detail_df.sort_values("y_prob", ascending=False).head(20)
top20_hit_rate = (top20.y_true == 1).mean()
ratio = top20.criticality_score.mean() / detail_df.criticality_score.mean()
print(f"top-20 hit rate: {top20_hit_rate:.0%} | criticality_score ratio: {ratio:.1f}x")

top-20 hit rate: 100% | criticality_score ratio: 3.3x


**Vertical 6 — Tier 1 vs. Tier 2 ablation:**

In [ ]:
comparison = pd.DataFrame({
    "Tier 1 (LightGBM)": {"accuracy": 0.869, "precision": 0.863, "recall": 0.788, "f1": 0.824},
    "Tier 2 (GNN)": {
        "accuracy": overall["accuracy"], "precision": overall["precision"],
        "recall": overall["recall"], "f1": overall["f1"],
    },
})
print(comparison)

           Tier 1 (LightGBM)  Tier 2 (GNN)
accuracy               0.869      0.868173
precision              0.863      0.746981
recall                 0.788      0.979916
f1                     0.824      0.847739


In [ ]:
deg_idx = FEATURE_COLUMNS.index("degree_centrality")  # position within the original 14-feature order

@torch.no_grad()
def eval_with_centrality(model, graphs):
    model.eval()
    rows = []
    for g in graphs:
        data = g.to(device)
        probs = torch.sigmoid(model(data.x, data.edge_index)).cpu().numpy()
        rows.append(pd.DataFrame({
            "y_true": data.y.cpu().numpy(),
            "y_prob": probs,
            "degree_centrality": data.x[:, deg_idx].cpu().numpy(),
        }))
    return pd.concat(rows, ignore_index=True)

decoy_df = eval_with_centrality(model, test_graphs)
decoy_df["y_pred"] = (decoy_df.y_prob >= threshold).astype(int)

high_centrality = decoy_df.degree_centrality >= decoy_df.degree_centrality.quantile(0.75)
decoys = decoy_df[(decoy_df.y_true == 0) & high_centrality]            # look important, aren't
hard_positives = decoy_df[(decoy_df.y_true == 1) & ~high_centrality]   # are important, don't look it
all_negatives = decoy_df[decoy_df.y_true == 0]
all_positives = decoy_df[decoy_df.y_true == 1]

print(f"decoys (high centrality, not critical): {len(decoys)} agents")
print(f"  false-positive rate on decoys:      {(decoys.y_pred == 1).mean():.1%}")
print(f"  false-positive rate on negatives overall: {(all_negatives.y_pred == 1).mean():.1%}")
print()
print(f"hard positives (low centrality, actually critical): {len(hard_positives)} agents")
print(f"  recall on hard positives: {(hard_positives.y_pred == 1).mean():.1%}")
print(f"  recall on positives overall: {(all_positives.y_pred == 1).mean():.1%}")

decoys (high centrality, not critical): 30352 agents
  false-positive rate on decoys:      24.2%
  false-positive rate on negatives overall: 19.9%

hard positives (low centrality, actually critical): 53170 agents
  recall on hard positives: 98.9%
  recall on positives overall: 98.0%
